# Extract DEX prices from Dune

Pull swap-level mid prices for one token pair on one blockchain, from each supported DEX, over a **collection window**, then save one CSV per DEX under `<chain>/`.

All reusable logic lives in the `arblib` package; this notebook only sets parameters and wires the steps together.

In [ ]:
# !pip install -r requirements.txt

In [6]:
from arblib.config import QUERY_IDS, TOKENS, build_collection_params
from arblib.dune_api import make_headers, run_dune_saved_query
from arblib.data_io import save_dataframes
import os
from pathlib import Path


## Parameters

- **CHAIN / tokens** — which market to pull.
- **Collection window** (`START_TS` / `END_TS`, UTC) — note `START_TS` is deliberately *earlier* than the study start used in `arbitrage.ipynb`, so every pool already has a known price to forward-fill from once the study window begins.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # reads .env if present, falls back to real env vars
DUNE_API_KEY = os.environ["DUNE_API_KEY"]

current = Path.cwd()
while current.name != 'defi_arbitrage' and current != current.parent:
    current = current.parent
BASE_DIR = current if current.name == 'defi_arbitrage' else Path.cwd()

# --- What to collect ------------------------------------------------
CHAIN  = "base"                 # blockchain name
TOKEN0 = TOKENS[CHAIN]["WETH"]  # base token
TOKEN1 = TOKENS[CHAIN]["USDC"]  # quote token

# --- Collection window (UTC) ---------------------------------------
START_TS = "2024-12-31 15:00:00"
END_TS   = "2025-01-01 12:00:00"

params  = build_collection_params(CHAIN, TOKEN0, TOKEN1, START_TS, END_TS)
headers = make_headers(DUNE_API_KEY)
params

## Run the saved queries

One query per DEX. A DEX not available on `CHAIN` (e.g. Aerodrome off Base) returns an empty DataFrame and is skipped on save.

In [3]:
df_uniswap   = run_dune_saved_query(QUERY_IDS["uniswap"],   params, headers, "Uniswap")
df_pancake   = run_dune_saved_query(QUERY_IDS["pancake"],   params, headers, "Pancake")
df_aerodrome = run_dune_saved_query(QUERY_IDS["aerodrome"], params, headers, "Aerodrome")

[Uniswap] EXECUTE RESPONSE: {'execution_id': '01KTRPSF5SDDD03P385J58AP7V', 'state': 'QUERY_STATE_PENDING'}
[Uniswap] STATUS: QUERY_STATE_PENDING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_COMPLETED
[Pancake] EXECUTE RESPONSE: {'execution_id': '01KTRPT2MFDN8955JN01QDMXKK', 'state': 'QUERY_STATE_PENDING'}
[Pancake] STATUS: QUERY_STATE_PENDING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_COMPLETED
[Aerodrome] EXECUTE RESPONSE: {'execution_id': '01KTRPTFSH846D2YQ4WP62Z8PR', 'state': 'QUERY_STATE_PENDING'}
[Aerodrome] STATUS: QUERY_STATE_PENDING
[Aerodrome] STATUS: QUERY_STATE_EXECUTING
[Aerodrome] STATUS: QUERY_STATE_EXECUTING
[Aerodrome] STATUS: QUERY_STATE_EXECUTING
[Aerodrome] STATUS: QUERY_STATE_COM

## Save one CSV per DEX under `<chain>/`

In [8]:

save_dir = os.path.join(BASE_DIR, CHAIN)

save_dataframes(
    {
        "df_uniswap.csv":   df_uniswap,
        "df_pancake.csv":   df_pancake,
        "df_aerodrome.csv": df_aerodrome,
    },
    save_dir,
)

Saved: /Users/matthieu/Downloads/defi_arbitrage/base/df_uniswap.csv
Saved: /Users/matthieu/Downloads/defi_arbitrage/base/df_pancake.csv
Saved: /Users/matthieu/Downloads/defi_arbitrage/base/df_aerodrome.csv
Done.
